In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
blocks = catalog.load_table("raw.blocks")
blocks.inspect.history().to_pandas()

,made_current_at,snapshot_id,parent_id,is_current_ancestor
0,2026-11-13 14:52:42.933,6085764886873205965,NaN,True
1,2026-11-13 14:52:54.800,7486038351758429705,6.085765e+18,True
2,2026-11-13 14:52:57.342,1477712972692656621,7.486038e+18,True
3,2026-11-13 14:53:00.071,5353843744686007607,1.477713e+18,True
4,2026-11-13 14:53:01.457,8545048210591028147,5.353844e+18,True
5,2026-11-13 14:53:08.371,8476948929999947215,8.545048e+18,True
6,2026-11-13 14:53:08.535,5265765793357399537,8.476949e+18,True
7,2026-11-13 14:53:08.742,8881365492073602260,5.265766e+18,True
8,2026-11-13 14:53:17.843,4247450943225277899,8.881365e+18,True
9,2026-11-13 14:54:14.048,1268416544308643856,4.247451e+18,True


In [3]:
for s in blocks.snapshots():
    print(s.snapshot_id, s.summary.operation, s.summary.get("total-records"))

6085764886873205965 Operation.APPEND 7177
7486038351758429705 Operation.APPEND 222412
1477712972692656621 Operation.APPEND 437445
5353843744686007607 Operation.APPEND 659896
8545048210591028147 Operation.APPEND 667072
8476948929999947215 Operation.OVERWRITE 666945
5265765793357399537 Operation.DELETE 659769
8881365492073602260 Operation.APPEND 666945
4247450943225277899 Operation.APPEND 674121
1268416544308643856 Operation.APPEND 681303


In [4]:
before_delete = blocks.snapshots()[4].snapshot_id  # the append of 1 Nov, just before we deleted empty blocks
blocks.scan(snapshot_id=before_delete).to_arrow().num_rows, blocks.scan().to_arrow().num_rows

(667072, 681303)

In [5]:
import datetime
blocks.scan(snapshot_id=blocks.snapshot_as_of_timestamp(blocks.snapshots()[3].timestamp_ms).snapshot_id).to_arrow().num_rows

659896

In [6]:
current = blocks.current_snapshot().snapshot_id
blocks.manage_snapshots().rollback_to_snapshot(before_delete).commit()
blocks.scan().to_arrow().num_rows

667072

In [7]:
blocks.manage_snapshots().set_current_snapshot(snapshot_id=current).commit()
blocks.scan().to_arrow().num_rows

681303